# Section 1: Setup

## Libraries and Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

import kagglehub
import os

In [2]:
train_df = pd.read_csv('/kaggle/input/sep-25-dl-gen-ai-nppe-2/train.csv')
test_df = pd.read_csv('/kaggle/input/sep-25-dl-gen-ai-nppe-2/test.csv')

sample_df = pd.read_csv('/kaggle/input/sep-25-dl-gen-ai-nppe-2/sample_submission.csv')

In [3]:
print(f"Training dataset shape: {train_df.shape}")
print(f"Test dataset shape: {test_df.shape}")

Training dataset shape: (7262, 4)
Test dataset shape: (1816, 2)


In [4]:
print(f"Unique columns of training set: {list(train_df.columns)}")
print("\n")
print(f"Unique columns of test set: {list(test_df.columns)}")

Unique columns of training set: ['id', 'seq', 'sst8', 'sst3']


Unique columns of test set: ['id', 'seq']


In [5]:
train_df.head(3)

,id,seq,sst8,sst3
0,0,GVGLEGGVQLSPARTRGPEFAAPEQAG,CCCCCCCCSCCCCCCGGGCCCCCCCCC,CCCCCCCCCCCCCCCHHHCCCCCCCCC
1,1,NHGKVKIEHTKWNVEYKVTYNRNVFANHIRSGELASNGYHTTRRTA...,CEEEEEECCTTTEEEEEEEEEEEEEEEEEEEEECSCCSSSCCCCEE...,CEEEEEECCCCCEEEEEEEEEEEEEEEEEEEEECCCCCCCCCCCEE...
2,2,EMRKMLADWKGLSKSDGMLSSEGRTKALWLGEANFSYVPKLDPRAS...,CTHHHHHHHHHSGGGCCCCCCCCCCCCEEECSSCEEEEEEETTCGG...,CCHHHHHHHHHCHHHCCCCCCCCCCCCEEECCCCEEEEEEECCCHH...


In [6]:
test_df.head(3)

,id,seq
0,0,FFKGSYQKVSNQLLYQANQIQDQTGTITIIRDESGELPEDIKISAG...
1,1,ATTKKNSPFPKVEEAYVSGDANITLFIKRGAHIAQNISSPYVGLDK...
2,2,VRALMLELRSGVREALDALGGVWEITKYLFMVDVPNLESELAFLQR...


## Mapping and Vocabulary building

In [7]:
# BUILDING AMINO ACID LIBRARY

seq = pd.concat([train_df["seq"],test_df["seq"]])
# seq.shape

amino_acids = set("".join(seq))

print(f"All Amino acids: {list(amino_acids)}")
print(f"\nShape: {len(amino_acids)}")

# 20 amino acids and "*" for all nonstandard residues — B, O, U, X, and Z

All Amino acids: ['Q', 'E', 'N', 'K', 'H', 'A', 'F', 'P', 'L', 'R', 'C', 'D', 'T', 'V', 'M', 'G', 'W', 'S', '*', 'Y', 'I']

Shape: 21


In [8]:
# Building Vocabulary mapping

PAD_TOKEN = "<PAD>"

amino_vocab = {PAD_TOKEN: 0}


for index, amino in enumerate(sorted(amino_acids), start=1):
    amino_vocab[amino] = index


# Reverse mapping (useful for debugging)
index_to_amino = {v: k for k, v in amino_vocab.items()}

print(amino_vocab)
print(f"\n {index_to_amino}")

{'<PAD>': 0, '*': 1, 'A': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7, 'H': 8, 'I': 9, 'K': 10, 'L': 11, 'M': 12, 'N': 13, 'P': 14, 'Q': 15, 'R': 16, 'S': 17, 'T': 18, 'V': 19, 'W': 20, 'Y': 21}

 {0: '<PAD>', 1: '*', 2: 'A', 3: 'C', 4: 'D', 5: 'E', 6: 'F', 7: 'G', 8: 'H', 9: 'I', 10: 'K', 11: 'L', 12: 'M', 13: 'N', 14: 'P', 15: 'Q', 16: 'R', 17: 'S', 18: 'T', 19: 'V', 20: 'W', 21: 'Y'}


In [9]:
# Quick checking 

assert amino_vocab["<PAD>"] == 0
assert "*" in amino_vocab
assert len(amino_vocab) == len(amino_acids) + 1


In [10]:
# Q8 Label mapping

q8_labels = sorted(set("".join(train_df["sst8"])))

print("Q8 labels:", q8_labels)
print("Count:", len(q8_labels))

q8_to_idx = {label: idx for idx, label in enumerate(q8_labels)}
idx_to_q8 = {idx: label for label, idx in q8_to_idx.items()}

q8_to_idx

Q8 labels: ['B', 'C', 'E', 'G', 'H', 'I', 'S', 'T']
Count: 8


{'B': 0, 'C': 1, 'E': 2, 'G': 3, 'H': 4, 'I': 5, 'S': 6, 'T': 7}

In [11]:
# Q3 label mapping

q3_labels = sorted(set("".join(train_df["sst3"])))

print("Q3 labels:", q3_labels)
print("Count:", len(q3_labels))

q3_to_idx = {label: idx for idx, label in enumerate(q3_labels)}
idx_to_q3 = {idx: label for label, idx in q3_to_idx.items()}

q3_to_idx

Q3 labels: ['C', 'E', 'H']
Count: 3


{'C': 0, 'E': 1, 'H': 2}

In [12]:
# Manual verification

row = train_df.iloc[0]

encoded_input = [amino_vocab[ch] for ch in row["seq"]]
encoded_q8    = [q8_to_idx[ch] for ch in row["sst8"]]
encoded_q3    = [q3_to_idx[ch] for ch in row["sst3"]]

print("Sequence length :", len(row["seq"]))
print("Raw input (seq)       :", row["seq"])
print("Encoded input length  :", len(encoded_input), " Encoded input :", encoded_input)
print("Raw Q8                :", row["sst8"])
print("Encoded Q8 length     :", len(encoded_q8)   , " Encoded Q8    :", encoded_q8)
print("Raw Q3                :", row["sst3"])
print("Encoded Q3 length     :", len(encoded_q3)   , " Encoded Q3    :", encoded_q3)


Sequence length : 27
Raw input (seq)       : GVGLEGGVQLSPARTRGPEFAAPEQAG
Encoded input length  : 27  Encoded input : [7, 19, 7, 11, 5, 7, 7, 19, 15, 11, 17, 14, 2, 16, 18, 16, 7, 14, 5, 6, 2, 2, 14, 5, 15, 2, 7]
Raw Q8                : CCCCCCCCSCCCCCCGGGCCCCCCCCC
Encoded Q8 length     : 27  Encoded Q8    : [1, 1, 1, 1, 1, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Raw Q3                : CCCCCCCCCCCCCCCHHHCCCCCCCCC
Encoded Q3 length     : 27  Encoded Q3    : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## Dataset and Dataloaders

In [13]:
# Dataset class

class ProteinDataset(Dataset):
    def __init__(self, df, amino_vocab, q8_to_idx=None, q3_to_idx=None):
        self.df = df
        self.amino_vocab = amino_vocab
        self.q8_to_idx = q8_to_idx
        self.q3_to_idx = q3_to_idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Encode amino acid sequence
        seq = row["seq"]
        input_ids = [self.amino_vocab[ch] for ch in seq]

        item = {
            "input_ids": input_ids
        }

        # Only available for training
        if self.q8_to_idx is not None and self.q3_to_idx is not None:
            sst8 = row["sst8"]
            sst3 = row["sst3"]

            item["q8_labels"] = [self.q8_to_idx[ch] for ch in sst8]
            item["q3_labels"] = [self.q3_to_idx[ch] for ch in sst3]

        return item


In [14]:
import torch

def collate_fn(batch):
    """
    - Pads sequences to same length
    - Creates attention mask
    - Converts lists → tensors
    """
    
    batch_size = len(batch)

    # Find max sequence length in batch
    max_len = max( len( item["input_ids"] ) for item in batch )

    
    input_ids = torch.zeros((batch_size, max_len), dtype=torch.long)
    attention_mask = torch.zeros((batch_size, max_len), dtype=torch.long)

    has_labels = "q8_labels" in batch[0]

    if has_labels:
        q8_labels = torch.zeros((batch_size, max_len), dtype=torch.long)
        q3_labels = torch.zeros((batch_size, max_len), dtype=torch.long)

    for i, item in enumerate(batch):
        seq_len = len(item["input_ids"])

        input_ids[i, :seq_len] = torch.tensor(item["input_ids"])
        attention_mask[i, :seq_len] = 1

        if has_labels:
            q8_labels[i, :seq_len] = torch.tensor(item["q8_labels"])
            q3_labels[i, :seq_len] = torch.tensor(item["q3_labels"])

    output = {
        "input_ids": input_ids,
        "attention_mask": attention_mask
    }

    if has_labels:
        output["q8_labels"] = q8_labels
        output["q3_labels"] = q3_labels

    return output


In [15]:
train_dataset = ProteinDataset(
    train_df,
    amino_vocab,
    q8_to_idx=q8_to_idx,
    q3_to_idx=q3_to_idx
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

## Model Architecture


BI-DIRECTIONAL RNN 

- Scratch model implementation


        input_ids (B,L)
           ↓
        Embedding [V → D] (B,L,D) 
           ↓
        BiRNN (→, ←)
           ↓
        Concatenated hidden states (2H)
           ↓
        Linear head (Q8) (B,L,8)
        Linear head (Q3) (B,L,3)


In [16]:
class BiRNNModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=64,
        hidden_dim=128,
        num_layers=1,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        # Dual heads
        self.q8_head = nn.Linear(hidden_dim * 2, 8)
        self.q3_head = nn.Linear(hidden_dim * 2, 3)

    def forward(self, input_ids):
        """
        input_ids: (B, L)
        """

        x = self.embedding(input_ids)          # (B, L, D)
        x, _ = self.rnn(x)                      # (B, L, 2H)
        x = self.dropout(x)

        q8_logits = self.q8_head(x)             # (B, L, 8)
        q3_logits = self.q3_head(x)             # (B, L, 3)

        return q8_logits, q3_logits


BiLSTM model

- Long-range dependencies
- Secondary structure depends on both sides
- LSTM gates prevent vanishing gradients


In [17]:
class BiLSTMModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=256,
        hidden_dim=384,
        num_layers=2,
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        self.dropout = nn.Dropout(dropout)

        self.q8_head = nn.Linear(hidden_dim * 2, 8)
        self.q3_head = nn.Linear(hidden_dim * 2, 3)

    def forward(self, input_ids):
        x = self.embedding(input_ids)           # (B, L, D)
        x, _ = self.lstm(x)                     # (B, L, 2H)
        x = self.dropout(x)

        q8_logits = self.q8_head(x)             # (B, L, 8)
        q3_logits = self.q3_head(x)             # (B, L, 3)

        return q8_logits, q3_logits


## Loss function, Optimizer and Metrics

In [18]:
# Loss Setup

PAD_IDX = 0

loss_q8_fn = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
loss_q3_fn = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

In [19]:
# Loss Computation

def compute_loss(q8_logits, q3_logits, q8_labels, q3_labels):
    # transpose: (B, L, C) → (B, C, L)
    q8_logits = q8_logits.permute(0, 2, 1)
    q3_logits = q3_logits.permute(0, 2, 1)

    loss_q8 = loss_q8_fn(q8_logits, q8_labels)
    loss_q3 = loss_q3_fn(q3_logits, q3_labels)

    total_loss = loss_q8 + loss_q3
    return total_loss, loss_q8, loss_q3

In [20]:
# Masked Token level F1

from sklearn.metrics import f1_score

def masked_f1(logits, labels, mask, num_classes):
    """
    logits: (B, L, C)
    labels: (B, L)
    mask:   (B, L)
    """

    preds = logits.argmax(dim=-1)  

    preds = preds.cpu().numpy().flatten()
    labels = labels.cpu().numpy().flatten()
    mask = mask.cpu().numpy().flatten()

    valid_idx = mask == 1

    preds = preds[valid_idx]
    labels = labels[valid_idx]

    return f1_score(labels, preds, average="macro")


In [21]:
# Harmonic mean (Kaggle Score)

def kaggle_score(f1_q8, f1_q3):
    return (2 * f1_q8 * f1_q3) / (f1_q8 + f1_q3 + 1e-8)

## Training loop

In [22]:
# One training epoch

def train_one_epoch(
    model,
    dataloader,
    optimizer
):
    model.train()

    total_loss = 0
    total_f1_q8 = 0
    total_f1_q3 = 0
    total_score = 0
    steps = 0

    for batch in dataloader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        q8_labels = batch["q8_labels"].to(device)
        q3_labels = batch["q3_labels"].to(device)

        q8_logits, q3_logits = model(input_ids)

        loss, _, _ = compute_loss(
            q8_logits,
            q3_logits,
            q8_labels,
            q3_labels
        )

        loss.backward()
        optimizer.step()

        with torch.no_grad():
            f1_q8 = masked_f1(q8_logits, q8_labels, attention_mask, 8)
            f1_q3 = masked_f1(q3_logits, q3_labels, attention_mask, 3)
            score = kaggle_score(f1_q8, f1_q3)

        total_loss += loss.item()
        total_f1_q8 += f1_q8
        total_f1_q3 += f1_q3
        total_score += score
        steps += 1

    metrics = {
        "train/loss": total_loss / steps,
        "train/f1_q8": total_f1_q8 / steps,
        "train/f1_q3": total_f1_q3 / steps,
        "train/kaggle_score": total_score / steps
    }

    
    trackio.log(metrics)

    return metrics


In [23]:
# Validation epoch

@torch.no_grad()
def validate_one_epoch(model, dataloader):
    model.eval()

    total_f1_q8 = 0
    total_f1_q3 = 0
    total_score = 0
    steps = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        q8_labels = batch["q8_labels"].to(device)
        q3_labels = batch["q3_labels"].to(device)

        q8_logits, q3_logits = model(input_ids)

        f1_q8 = masked_f1(q8_logits, q8_labels, attention_mask, 8)
        f1_q3 = masked_f1(q3_logits, q3_labels, attention_mask, 3)
        score = kaggle_score(f1_q8, f1_q3)

        total_f1_q8 += f1_q8
        total_f1_q3 += f1_q3
        total_score += score
        steps += 1

    metrics = {
        "val/f1_q8": total_f1_q8 / steps,
        "val/f1_q3": total_f1_q3 / steps,
        "val/kaggle_score": total_score / steps
    }

    
    trackio.log(metrics)

    return metrics

## Trackio Initialisation

In [24]:
# !pip install trackio 

# #Successfully installed gradio-6.1.0 gradio-client-2.0.1 plotly-6.5.0 safehttpx-0.1.7 trackio-0.13.0

In [25]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()

hf_token  = user_secrets.get_secret("HuggingFaceToken")
os.environ['HF_TOKEN '] = hf_token

In [26]:
# ============================================================
# TRACKIO INITIALIZATION
# ============================================================

# import trackio

# trackio.init(
#     project="25-t3-nppe2",        
#     name="Bi-directional RNN baseline model",              
#     group="Bidirectional RNN",               
#     space_id="Niraj-DS/dlgenai-nppe"
# )

# Training Code

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


Bi-RNN Model Training.

In [28]:
# birnn = BiRNNModel(vocab_size=len(amino_vocab)).to(device)
# optimizer = torch.optim.Adam(birnn.parameters(), lr=1e-3)

# best_score = 0
# EPOCHS = 5

# for epoch in range(EPOCHS):
#     train_metrics = train_one_epoch(
#         birnn,
#         train_loader,
#         optimizer
#     )

#     val_metrics = validate_one_epoch(
#         birnn,
#         train_loader
#     )

#     print(f"[BiRNN][Epoch {epoch+1}] {val_metrics}")

#     if val_metrics["val/kaggle_score"] > best_score:
#         best_score = val_metrics["val/kaggle_score"]
#         torch.save(birnn.state_dict(), "birnn_best.pt")


In [29]:
# ============================================================
# TRACKIO LOGGING FINISHED - BI DIRECTIONAL RNN (SCRATCH MODEL)
# ============================================================

# trackio.finish()

Bi-LSTM Model Training

In [30]:
# ============================================================
# TRACKIO INITIALIZATION
# ============================================================


# trackio.init(
#     project="25-t3-nppe2",
#     name="BiLSTM model",
#     group="BiLSTM",
#     space_id="Niraj-DS/dlgenai-nppe"
# )


In [31]:
bilstm = BiLSTMModel(
    vocab_size=len(amino_vocab),
    embed_dim=128,
    hidden_dim=256,
    num_layers=2,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(bilstm.parameters(), lr=3e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=1, verbose=True
)

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [32]:
# best_score = 0.0
# EPOCHS = 25

# for epoch in range(EPOCHS):
#     train_metrics = train_one_epoch(
#         bilstm,
#         train_loader,
#         optimizer
#     )

#     val_metrics = validate_one_epoch(
#         bilstm,
#         train_loader  
#     )

#     scheduler.step(val_metrics["val/kaggle_score"])

#     print(f"[BiLSTM][Epoch {epoch+1}] {val_metrics}")

#     # Save best checkpoint
#     if val_metrics["val/kaggle_score"] > best_score:
#         best_score = val_metrics["val/kaggle_score"]
#         torch.save(bilstm.state_dict(), "bilstm_best.pt")

# print("Training finished")

# trackio.finish()

# Hyper Tuning and different models approach.

In this section, we will implement various strategies for preprocessing and model building for better generalization. 
In our last attempt, we only focused on getting the rubrics right. 

## Train test split

In [34]:
from sklearn.model_selection import train_test_split

train_df_split, val_df_split = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Train size:", len(train_df_split))
print("Validation size:", len(val_df_split))

Train size: 5809
Validation size: 1453


In [35]:
# Dataset objects

train_dataset = ProteinDataset(
    train_df_split,
    amino_vocab,
    q8_to_idx=q8_to_idx,
    q3_to_idx=q3_to_idx
)

val_dataset = ProteinDataset(
    val_df_split,
    amino_vocab,
    q8_to_idx=q8_to_idx,
    q3_to_idx=q3_to_idx
)

In [36]:
# Dataloaders 

from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,        
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,        
    collate_fn=collate_fn
)


## BiGRU Implementation

In [37]:
import torch
import torch.nn as nn

class BiGRUModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=128,
        hidden_dim=192,
        num_layers=2,
        dropout=0.3
    ):
        super().__init__()

        # Amino acid embedding
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        # Bidirectional GRU
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)

        # Output heads
        self.q8_head = nn.Linear(hidden_dim * 2, 8)
        self.q3_head = nn.Linear(hidden_dim * 2, 3)

    def forward(self, input_ids):
        """
        input_ids: (B, L)
        """

        x = self.embedding(input_ids)     # (B, L, D)
        x, _ = self.gru(x)                # (B, L, 2H)
        x = self.dropout(x)

        q8_logits = self.q8_head(x)       # (B, L, 8)
        q3_logits = self.q3_head(x)       # (B, L, 3)

        return q8_logits, q3_logits


In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bigru = BiGRUModel(
    vocab_size=len(amino_vocab),
    embed_dim=128,
    hidden_dim=192,   
    num_layers=2,
    dropout=0.3
).to(device)



optimizer = torch.optim.Adam(
    bigru.parameters(),
    lr=3e-4,
    weight_decay=1e-4   
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
    verbose=True
)


In [39]:
# ============================================================
# TRACKIO INITIALIZATION
# ============================================================



# import trackio

# trackio.init(
#     project="25-t3-nppe2",        
#     name="Bi-directional GRU run 1",              
#     group="BiGRU RNN",               
#     space_id="Niraj-DS/dlgenai-nppe"
# )

In [40]:
# best_val_score = 0.0
# patience = 5
# patience_counter = 0
# EPOCHS = 40



# for epoch in range(EPOCHS):
#     # -------- TRAIN --------
#     train_metrics = train_one_epoch(
#         bigru,
#         train_loader,
#         optimizer
#     )

#     # -------- VALIDATE --------
#     val_metrics = validate_one_epoch(
#         bigru,
#         val_loader
#     )

#     val_score = val_metrics["val/kaggle_score"]
#     scheduler.step(val_score)

#     # -------- TRACKIO LOGGING --------
#     trackio.log({
#         "epoch": epoch + 1,
#         **train_metrics,
#         **val_metrics
#     })

#     print(f"[BiGRU][Epoch {epoch+1}] {val_metrics}")

#     # -------- SAVE BEST MODEL --------
#     if val_score > best_val_score:
#         best_val_score = val_score
#         patience_counter = 0
#         torch.save(bigru.state_dict(), "bigru_best.pt")
#         print(f"✅ New best BiGRU model saved (score={val_score:.4f})")
#     else:
#         patience_counter += 1
#         print(f"⏳ No improvement ({patience_counter}/{patience})")

#     # -------- EARLY STOPPING --------
#     if patience_counter >= patience:
#         print("🛑 Early stopping triggered")
#         break

# trackio.finish()

## Implementing weight class

In [42]:
from collections import Counter

q8_counter = Counter()
q3_counter = Counter()

for _, row in train_df_split.iterrows():
    q8_counter.update(list(row["sst8"]))
    q3_counter.update(list(row["sst3"]))

print("Q8 counts:", q8_counter)
print("Q3 counts:", q3_counter)


Q8 counts: Counter({'H': 447844, 'C': 343675, 'E': 297577, 'T': 151113, 'S': 109856, 'G': 53591, 'B': 14947, 'I': 253})
Q3 counts: Counter({'C': 604644, 'H': 501688, 'E': 312524})


In [43]:
# Q8 weights
q8_weights = torch.zeros(len(q8_to_idx))
for label, idx in q8_to_idx.items():
    q8_weights[idx] = 1.0 / q8_counter[label]

q8_weights = q8_weights / q8_weights.sum()

# Q3 weights
q3_weights = torch.zeros(len(q3_to_idx))
for label, idx in q3_to_idx.items():
    q3_weights[idx] = 1.0 / q3_counter[label]

q3_weights = q3_weights / q3_weights.sum()

print("Q8 weights:", q8_weights)
print("\n")
print("Q3 weights:", q3_weights)


Q8 weights: tensor([1.6469e-02, 7.1627e-04, 8.2722e-04, 4.5934e-03, 5.4966e-04, 9.7297e-01,
        2.2408e-03, 1.6290e-03])


Q3 weights: tensor([0.2415, 0.4673, 0.2911])


In [44]:
# ============================================================
# GRU-SPECIFIC CLASS-WEIGHTED LOSS
# ============================================================

criterion_q8_gru = torch.nn.CrossEntropyLoss(
    weight=q8_weights.to(device),
    ignore_index=0
)

criterion_q3_gru = torch.nn.CrossEntropyLoss(
    weight=q3_weights.to(device),
    ignore_index=0
)

# rewriting our old compute_loss function to not mess up old runs
def compute_loss(q8_logits, q3_logits, q8_labels, q3_labels):
    """
    Loss function used ONLY for BiGRU training
    """

    loss_q8 = criterion_q8_gru(
        q8_logits.view(-1, 8),
        q8_labels.view(-1)
    )

    loss_q3 = criterion_q3_gru(
        q3_logits.view(-1, 3),
        q3_labels.view(-1)
    )

    loss = loss_q8 + loss_q3
    return loss, loss_q8, loss_q3


In [45]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bigru = BiGRUModel(
    vocab_size=len(amino_vocab),
    embed_dim=128,
    hidden_dim=192,   
    num_layers=2,
    dropout=0.3
).to(device)



optimizer = torch.optim.Adam(
    bigru.parameters(),
    lr=3e-4,
    weight_decay=1e-4   
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
    verbose=True
)


In [46]:
# # ============================================================
# # TRACKIO INITIALIZATION
# # ============================================================



# import trackio

# trackio.init(
#     project="25-t3-nppe2",        
#     name="Bi-directional class weighted GRU run 1",              
#     group="BiGRU RNN",               
#     space_id="Niraj-DS/dlgenai-nppe"
# )

In [47]:
# best_val_score = 0.0
# patience = 7
# patience_counter = 0
# EPOCHS = 40



# for epoch in range(EPOCHS):
#     # -------- TRAIN --------
#     train_metrics = train_one_epoch(
#         bigru,
#         train_loader,
#         optimizer
#     )

#     # -------- VALIDATE --------
#     val_metrics = validate_one_epoch(
#         bigru,
#         val_loader
#     )

#     val_score = val_metrics["val/kaggle_score"]
#     scheduler.step(val_score)

#     # -------- TRACKIO LOGGING --------
#     trackio.log({
#         "epoch": epoch + 1,
#         **train_metrics,
#         **val_metrics
#     })

#     print(f"[BiGRU][Epoch {epoch+1}] {val_metrics}")

#     # -------- SAVE BEST MODEL --------
#     if val_score > best_val_score:
#         best_val_score = val_score
#         patience_counter = 0
#         torch.save(bigru.state_dict(), "bigru_best.pt")
#         print(f"✅ New best BiGRU model saved (score={val_score:.4f})")
#     else:
#         patience_counter += 1
#         print(f"⏳ No improvement ({patience_counter}/{patience})")

#     # -------- EARLY STOPPING --------
#     if patience_counter >= patience:
#         print("🛑 Early stopping triggered")
#         break


# trackio.finish()

# Implementing CNN + BiGRU model for better results

In [49]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNBiGRUModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=128,
        cnn_channels=128,
        kernel_size=5,
        gru_hidden_dim=192,
        gru_layers=2,
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        # CNN for local patterns
        self.conv = nn.Conv1d(
            in_channels=embed_dim,
            out_channels=cnn_channels,
            kernel_size=kernel_size,
            padding=kernel_size // 2
        )

        self.conv_dropout = nn.Dropout(dropout)

        # BiGRU for global context
        self.bigru = nn.GRU(
            input_size=cnn_channels,
            hidden_size=gru_hidden_dim,
            num_layers=gru_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if gru_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)

        self.q8_head = nn.Linear(gru_hidden_dim * 2, 8)
        self.q3_head = nn.Linear(gru_hidden_dim * 2, 3)

    def forward(self, input_ids):
        # input_ids: (B, L)
        x = self.embedding(input_ids)          # (B, L, D)

        x = x.transpose(1, 2)                  # (B, D, L)
        x = F.relu(self.conv(x))               # (B, C, L)
        x = self.conv_dropout(x)
        x = x.transpose(1, 2)                  # (B, L, C)

        x, _ = self.bigru(x)                   # (B, L, 2H)
        x = self.dropout(x)

        q8_logits = self.q8_head(x)
        q3_logits = self.q3_head(x)

        return q8_logits, q3_logits


In [50]:
model = CNNBiGRUModel(
    vocab_size=len(amino_vocab),
    embed_dim=128,
    cnn_channels=128,
    kernel_size=5,
    gru_hidden_dim=192,
    gru_layers=2,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
    verbose=True
)


In [51]:
# ============================================================
# TRACKIO INITIALIZATION
# ============================================================


# import trackio

# trackio.init(
#     project="25-t3-nppe2",        
#     name="Bi-directional GRU + CNN model | run 1",              
#     group="BiGRU RNN + CNN",               
#     space_id="Niraj-DS/dlgenai-nppe"
# )

In [52]:
# best_val_score = 0.0
# patience = 5
# patience_counter = 0
# EPOCHS = 40

# for epoch in range(EPOCHS):

#     # -------- TRAIN --------
#     train_metrics = train_one_epoch(
#         model,              # CNN + BiGRU
#         train_loader,
#         optimizer
#     )

#     # -------- VALIDATE --------
#     val_metrics = validate_one_epoch(
#         model,              # CNN + BiGRU
#         val_loader
#     )

#     val_score = val_metrics["val/kaggle_score"]
#     scheduler.step(val_score)

#     # -------- TRACKIO LOGGING --------
#     trackio.log({
#         "epoch": epoch + 1,
#         **train_metrics,
#         **val_metrics
#     })

#     print(f"[CNN+BiGRU][Epoch {epoch+1}] {val_metrics}")

#     # -------- SAVE BEST MODEL --------
#     if val_score > best_val_score:
#         best_val_score = val_score
#         patience_counter = 0
#         torch.save(model.state_dict(), "cnn_bigru_best.pt")
#         print(f"✅ New best CNN+BiGRU model saved (score={val_score:.4f})")
#     else:
#         patience_counter += 1
#         print(f"⏳ No improvement ({patience_counter}/{patience})")

#     # -------- EARLY STOPPING --------
#     if patience_counter >= patience:
#         print("🛑 Early stopping triggered")
#         break


# trackio.finish()

## Multi-Kernel CNN + Residual + BiLSTM

In [54]:
# Cannot comment out installation of pytorch crf as it is used in the model definition

!pip install pytorch-crf --no-cache-dir
# !pip install trackio

In [67]:
q3_class_weights = torch.zeros(len(q3_to_idx), dtype=torch.float)

for label, idx in q3_to_idx.items():
    q3_class_weights[idx] = 1.0 / q3_counter[label]

q3_class_weights = q3_class_weights.to(device)  # moving to gpu

In [68]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchcrf import CRF

class MultiCNNBiLSTM_CRF(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=128,
        cnn_channels=128,
        lstm_hidden=256,
        lstm_layers=2,
        dropout=0.4
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # CNN block 1
        self.conv3_1 = nn.Conv1d(embed_dim, cnn_channels, 3, padding=1)
        self.conv5_1 = nn.Conv1d(embed_dim, cnn_channels, 5, padding=2)
        self.conv7_1 = nn.Conv1d(embed_dim, cnn_channels, 7, padding=3)
        self.bn1 = nn.BatchNorm1d(cnn_channels * 3)

        # CNN block 2
        self.conv3_2 = nn.Conv1d(cnn_channels * 3, cnn_channels, 3, padding=1)
        self.conv5_2 = nn.Conv1d(cnn_channels * 3, cnn_channels, 5, padding=2)
        self.conv7_2 = nn.Conv1d(cnn_channels * 3, cnn_channels, 7, padding=3)
        self.bn2 = nn.BatchNorm1d(cnn_channels * 3)

        self.dropout = nn.Dropout(dropout)

        self.bilstm = nn.LSTM(
            input_size=cnn_channels * 3 + embed_dim,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        # Q8 + CRF
        self.q8_emission = nn.Linear(lstm_hidden * 2, 8)
        self.crf = CRF(8, batch_first=True)

        # Q3
        self.q3_head = nn.Linear(lstm_hidden * 2, 3)

    def forward(self, input_ids):
        emb = self.embedding(input_ids)              # (B, L, D)
        x = emb.transpose(1, 2)                      # (B, D, L)

        c1 = torch.cat([
            self.conv3_1(x),
            self.conv5_1(x),
            self.conv7_1(x)
        ], dim=1)
        c1 = self.dropout(F.relu(self.bn1(c1)))

        c2 = torch.cat([
            self.conv3_2(c1),
            self.conv5_2(c1),
            self.conv7_2(c1)
        ], dim=1)
        c2 = self.dropout(F.relu(self.bn2(c2)))
        c2 = c2.transpose(1, 2)

        lstm_input = torch.cat([c2, emb], dim=-1)
        out, _ = self.bilstm(lstm_input)
        out = self.dropout(out)

        q8_emissions = self.q8_emission(out)
        q3_logits = self.q3_head(out)

        return q8_emissions, q3_logits

# importing CRF made me cry cuz it wasnt getting detected by kaggle....

In [69]:
# new loss function cuz its easier to create one than to fix the old one...

def compute_loss_crf(
    model,
    q8_emissions,
    q3_logits,
    q8_labels,
    q3_labels,
    attention_mask,
    q3_class_weights
):
    crf_loss = -model.crf(
        q8_emissions,
        q8_labels,
        mask=attention_mask.bool(),
        reduction="mean"
    )

    active = attention_mask.view(-1) == 1
    ce_q3 = nn.CrossEntropyLoss(weight=q3_class_weights)

    q3_loss = ce_q3(
        q3_logits.view(-1, 3)[active],
        q3_labels.view(-1)[active]
    )

    return crf_loss + q3_loss


In [70]:
from sklearn.metrics import f1_score

# masking before f1
def masked_f1_crf(preds, labels, mask, num_classes):
    true, pred = [], []

    for i, seq in enumerate(preds):
        length = mask[i].sum().item()
        true.extend(labels[i][:length].tolist())
        pred.extend(seq[:length])

    return f1_score(true, pred, average="macro")


In [74]:
# finally the model - MULTI CNN Bi-LSTM CRF ( I didnt have a better name in mind. My apologies!)
model = MultiCNNBiLSTM_CRF(
    vocab_size=len(amino_vocab),
    embed_dim=128,
    cnn_channels=128,
    lstm_hidden=256,
    lstm_layers=2,
    dropout=0.4
).to(device)


# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)


# Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

In [72]:
# Training loop for final model

def train_one_epoch(model, dataloader, optimizer, q3_class_weights):
    model.train()

    total_loss = total_f1_q8 = total_f1_q3 = total_score = 0
    steps = 0

    for batch in dataloader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        q8_labels = batch["q8_labels"].to(device)
        q3_labels = batch["q3_labels"].to(device)

        q8_emissions, q3_logits = model(input_ids)

        loss = compute_loss_crf(
            model,
            q8_emissions,
            q3_logits,
            q8_labels,
            q3_labels,
            attention_mask,
            q3_class_weights
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        with torch.no_grad():
            q8_preds = model.crf.decode(q8_emissions, mask=attention_mask.bool())
            f1_q8 = masked_f1_crf(q8_preds, q8_labels, attention_mask, 8)
            f1_q3 = masked_f1(q3_logits, q3_labels, attention_mask, 3)
            score = kaggle_score(f1_q8, f1_q3)

        total_loss += loss.item()
        total_f1_q8 += f1_q8
        total_f1_q3 += f1_q3
        total_score += score
        steps += 1

    return {
        "train/loss": total_loss / steps,
        "train/f1_q8": total_f1_q8 / steps,
        "train/f1_q3": total_f1_q3 / steps,
        "train/kaggle_score": total_score / steps
    }


# Validation function for final model
@torch.no_grad()
def validate_one_epoch(model, dataloader, q3_class_weights):
    model.eval()

    total_f1_q8 = total_f1_q3 = total_score = 0
    steps = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        q8_labels = batch["q8_labels"].to(device)
        q3_labels = batch["q3_labels"].to(device)

        q8_emissions, q3_logits = model(input_ids)
        q8_preds = model.crf.decode(q8_emissions, mask=attention_mask.bool())

        f1_q8 = masked_f1_crf(q8_preds, q8_labels, attention_mask, 8)
        f1_q3 = masked_f1(q3_logits, q3_labels, attention_mask, 3)
        score = kaggle_score(f1_q8, f1_q3)

        total_f1_q8 += f1_q8
        total_f1_q3 += f1_q3
        total_score += score
        steps += 1

    return {
        "val/f1_q8": total_f1_q8 / steps,
        "val/f1_q3": total_f1_q3 / steps,
        "val/kaggle_score": total_score / steps
    }


In [77]:
# ============================================================
# TRACKIO INITIALIZATION
# ============================================================

# import trackio
# trackio.init(
#     project="25-t3-nppe2",
#     name="MultiCNN-BiLSTM-CRF | baseline",
#     group="MultiCNN + BiLSTM + CRF",
#     space_id="Niraj-DS/dlgenai-nppe"
# )


* Trackio project initialized: 25-t3-nppe2
* Trackio metrics will be synced to Hugging Face Dataset: Niraj-DS/dlgenai-nppe-dataset
* Found existing space: https://huggingface.co/spaces/Niraj-DS/dlgenai-nppe
* View dashboard by going to: https://Niraj-DS-dlgenai-nppe.hf.space/


* Created new run: MultiCNN-BiLSTM-CRF | baseline


In [116]:
# best_val_score = 0.0
# patience = 12
# patience_counter = 0
# EPOCHS = 60

# for epoch in range(EPOCHS):
#     train_metrics = train_one_epoch(
#         model, train_loader, optimizer, q3_class_weights
#     )

#     val_metrics = validate_one_epoch(
#         model, val_loader, q3_class_weights
#     )

#     val_score = val_metrics["val/kaggle_score"]
#     scheduler.step(val_score)

#     # ✅ TrackIO logging (THIS is enough)
#     trackio.log({
#         "epoch": epoch + 1,
#         **train_metrics,
#         **val_metrics,
#         "lr": optimizer.param_groups[0]["lr"]
#     })

#     print(f"[CRF][Epoch {epoch+1}] {val_metrics}")

#     if val_score > best_val_score:
#         best_val_score = val_score
#         patience_counter = 0
#         torch.save(model.state_dict(), "best_multicnn_bilstm_crf.pt")
#         print(f"✅ New best model saved (score={val_score:.4f})")
#     else:
#         patience_counter += 1
#         print(f"⏳ No improvement ({patience_counter}/{patience})")

#     if patience_counter >= patience:
#         print("🛑 Early stopping triggered")
#         break

# trackio.finish()


# Inference Code

In [81]:
# Uploading the final model to kagglehub
import kagglehub

# model_path = "/kaggle/working/best_multicnn_bilstm_crf.pt"
# handle = "nirajds/MultiCNN_BiLSTM_CRF/pytorch/v1"

# kagglehub.model_upload(handle, 
#                        model_path, 
#                        version_notes = "MultiCNN + BiLSTM + CRF (Q8 structured prediction)")


Uploading Model https://www.kaggle.com/models/nirajds/MultiCNN_BiLSTM_CRF/pytorch/v1 ...
Model 'MultiCNN_BiLSTM_CRF' does not exist or access is forbidden for user 'nirajds'. Creating or handling Model...
Model 'MultiCNN_BiLSTM_CRF' Created.
Starting upload for file /kaggle/working/best_multicnn_bilstm_crf.pt


Uploading: 100%|██████████| 16.6M/16.6M [00:00<00:00, 21.4MB/s]

Upload successful: /kaggle/working/best_multicnn_bilstm_crf.pt (16MB)


Your model instance has been created.
Files are being processed...
See at: https://www.kaggle.com/models/nirajds/MultiCNN_BiLSTM_CRF/pytorch/v1


In [82]:
# Downloading the same model for inference

handle = "nirajds/MultiCNN_BiLSTM_CRF/pytorch/v1"
model_path = kagglehub.model_download(handle)
print(model_path)

model_file = os.path.join(model_path, "best_multicnn_bilstm_crf.pt")

/kaggle/input/multicnn_bilstm_crf/pytorch/v1/1


In [85]:
print(model_file)
print(f"Device in use: {device}")

/kaggle/input/multicnn_bilstm_crf/pytorch/v1/1/best_multicnn_bilstm_crf.pt
Device in use: cuda


## predicting the sst8 and sst3

In [86]:
# recreating the model with same parameters and loading the state from downloaded kagglehub model
submission_model = MultiCNNBiLSTM_CRF(
    vocab_size=len(amino_vocab),
    embed_dim=128,
    cnn_channels=128,
    lstm_hidden=256,
    lstm_layers=2,
    dropout=0.4
).to(device)

submission_model.load_state_dict(
    torch.load(model_file, map_location=device)
)

submission_model.eval()


MultiCNNBiLSTM_CRF(
  (embedding): Embedding(22, 128, padding_idx=0)
  (conv3_1): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv5_1): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,))
  (conv7_1): Conv1d(128, 128, kernel_size=(7,), stride=(1,), padding=(3,))
  (bn1): BatchNorm1d(384, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3_2): Conv1d(384, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv5_2): Conv1d(384, 128, kernel_size=(5,), stride=(1,), padding=(2,))
  (conv7_2): Conv1d(384, 128, kernel_size=(7,), stride=(1,), padding=(3,))
  (bn2): BatchNorm1d(384, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.4, inplace=False)
  (bilstm): LSTM(512, 256, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
  (q8_emission): Linear(in_features=512, out_features=8, bias=True)
  (crf): CRF(num_tags=8)
  (q3_head): Linear(in_features=512, out_features=3, bias=True)
)

In [87]:
test_dataset = ProteinDataset(
    test_df,
    amino_vocab,
    q8_to_idx=None,
    q3_to_idx=None
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)


In [113]:
final_q8 = []
final_q3 = []

submission_model.eval()

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        q8_emissions, q3_logits = submission_model(input_ids)

        # ---- Q8: CRF decode (already length-correct per sample)
        q8_preds = submission_model.crf.decode(
            q8_emissions,
            mask=attention_mask.bool()
        )

        # ---- Q3: per-sample argmax with masking
        for i in range(len(q8_preds)):
            seq_len = attention_mask[i].sum().item()

            # Q8 string
            q8_str = "".join(idx_to_q8[p] for p in q8_preds[i][:seq_len])

            # Q3 string (THIS WAS YOUR BUG)
            q3_pred = q3_logits[i][:seq_len].argmax(dim=-1).cpu().tolist()
            q3_str = "".join(idx_to_q3[p] for p in q3_pred)

            final_q8.append(q8_str)
            final_q3.append(q3_str)


In [114]:
idx_to_q8 = {v: k for k, v in q8_to_idx.items()}
idx_to_q3 = {v: k for k, v in q3_to_idx.items()}

In [115]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "sst8": final_q8,
    "sst3": final_q3
})
submission.to_csv("submission.csv", index=False)
print("submission.csv saved!")

submission.csv saved!
